# 02 — Resultados de la etapa 1 (zero-shot, VAL)

Análisis de la evaluación zero-shot de la etapa 1 (CLAUDE.md §3; `docs/metodologia.md` §2.2). **Solo lee
`runs/`**: no infiere ni entrena. Usa la misma implementación del extractor y de las métricas que
`scripts/score.py` (`vlmfid.eval`), así que los números deben coincidir con `docs/resultados_etapa1.md`.

**Todo es VAL.** Test no se toca en la etapa 1 (solo `scripts/final_test.py` lo lee).

| § | Pregunta |
|---|---|
| 1 | Líneas base sin modelo (clase mayoritaria, techo del color) |
| 2 | Exactitud por atributo, modelo × prompt (NF4) |
| 3 | Selección de prompt y sensibilidad al prompt; ¿es robusta la elección? (bootstrap pareado) |
| 4 | Omitir frente a equivocarse: formato o percepción |
| 5 | Matrices de confusión de forma, x e y |
| 6 | Color: errores más frecuentes |
| 7 | Control bf16: NF4 frente a bf16 por imagen (McNemar) |
| 8 | Métricas de texto frente a fidelidad por atributo |
| 9 | Piloto (200) frente a val completo: ¿el piloto anticipaba la elección? |
| 10 | Y-Decoder genérico (MSR-VTT): barrido de *k* y techo en M3DI |
| 11 | VL-JEPA zero-shot, condición intermedia, diagnóstico de cuantización y pérdida de información (si existen) |
| 12 | Tabla resumen para el informe |

**Requisitos (cluster):** `scripts/score.py` ya corrido sobre los runs (escribe `metrics.json` con las métricas de
texto, que necesitan Java). Kernel `vlmfid` (ver `notebooks/README.md`). Las secciones de VL-JEPA se saltan solas
si sus runs no existen todavía: el notebook se puede volver a ejecutar a medida que avanzan las colas.

> **Cierre de la etapa 1 (CE1, decisión del usuario, 20-11-2026; `docs/metodologia.md` §6.1).** La etapa se
> cerró sin más entrenamientos. De VL-JEPA hay: zero-shot NF4 en val con los 6 prompts (§11a), el Y-Decoder
> genérico NF4 (§10) y el diagnóstico de cuantización (§11c). **No hay** control bf16 de VL-JEPA, condición
> intermedia ni pérdida de información de la etapa 1: §11b y §11d quedan vacías a propósito y así se declara
> en el informe.


In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

from vlmfid.data.manifest import load_manifest
from vlmfid.eval import ATTRIBUTES, METRICS, TEXT_METRICS, color_ceiling, extract, majority_baseline, score, truth_from_record
from vlmfid.eval.attributes import json_mode_for
from vlmfid.paths import RUNS
from vlmfid.reports import save_fig, save_table

pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

MODELS = {"llava_ov_05b": "LLaVA-OV-0.5B", "internvl3_1b": "InternVL3-1B",
          "qwen35_08b_base": "Qwen3.5-0.8B-Base", "vljepa_open_msrvtt": "VL-JEPA"}
VLMS = ["llava_ov_05b", "internvl3_1b", "qwen35_08b_base"]
PROMPTS = ["p0_minimal", "p1_attributes", "p2_constrained", "p3_json", "p4_dataset_format", "p4m_multiexample"]
ATTR_ES = {"shape": "forma", "x": "x", "y": "y", "color": "color", "pos": "posición (x e y)", "attr_mean": "attr_mean"}
RNG = np.random.default_rng(0)

# Figuras y tablas del informe -> reports/figures/etapa1/ y reports/tables/etapa1/ (versionadas; reports/README.md).
# SAVE = False para explorar sin tocar reports/.
STAGE, SAVE = "etapa1", True


def show(name, fig=None, source=""):
    """plt.show() que además guarda la figura (PNG y PDF) en reports/figures/etapa1/<name>."""
    fig = fig or plt.gcf()
    if SAVE:
        save_fig(fig, STAGE, name, source=f"notebook 02 {source}".strip())
    plt.show()


def keep(df, name, source=""):
    """Guarda una tabla resumen en reports/tables/etapa1/<name>.csv y la devuelve (para display)."""
    if SAVE and df is not None and len(df):
        save_table(df, STAGE, name, source=f"notebook 02 {source}".strip())
    return df


print("runs:", RUNS, "| guardar en reports/:", SAVE)

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass                                    # línea truncada por una caída
    return rows


def load_run(run_dir):
    """Una fila por imagen: verdad, atributos extraídos y aciertos (misma lógica que scripts/score.py)."""
    run_dir = Path(run_dir)
    cfg = OmegaConf.load(run_dir / "config.yaml")
    if str(cfg.data.split) != "val":
        raise ValueError(f"{run_dir.name}: split {cfg.data.split}; este notebook es solo VAL")
    jm = json_mode_for(cfg)
    rows, seen = [], set()
    for r in read_jsonl(run_dir / "predictions.jsonl"):
        if r["id"] in seen:                             # una reanudación puede repetir la última muestra
            continue
        seen.add(r["id"])
        t, e = truth_from_record(r), extract(r["prediction"], jm)
        s = score(e, t)
        rows.append({"id": r["id"], "prediction": r["prediction"], "reference": r["reference"],
                     **{f"true_{a}": t[a] for a in ATTRIBUTES}, **{f"pred_{a}": e[a] for a in ATTRIBUTES},
                     **{f"ok_{m}": float(s[m]) for m in METRICS}, "n_words": e["n_words"]})
    df = pd.DataFrame(rows).set_index("id")
    df.attrs = {"model": str(cfg.model.name), "quant": str(cfg.model.quant), "prompt": str(cfg.prompt.id),
                "exp_id": str(cfg.exp_id), "json_mode": jm}
    return df


def run_metrics(run_dir):
    """metrics.json de scripts/score.py (incluye las métricas de texto), o None."""
    p = Path(run_dir) / "metrics.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None


def summary_row(df):
    """Exactitud media por métrica, omisiones y largo medio de un run cargado con load_run."""
    out = {"model": df.attrs["model"], "quant": df.attrs["quant"], "prompt": df.attrs["prompt"], "n": len(df)}
    out |= {m: df[f"ok_{m}"].mean() for m in METRICS}
    out |= {f"omit_{a}": df[f"pred_{a}"].isna().mean() for a in ATTRIBUTES}
    out["words"] = df["n_words"].mean()
    return out


def paired_bootstrap(a, b, n_boot=2000):
    """Diferencia media a − b sobre las mismas imágenes e IC 95 % por bootstrap pareado."""
    d = (a - b).to_numpy()
    idx = RNG.integers(0, len(d), size=(n_boot, len(d)))
    boots = d[idx].mean(axis=1)
    return d.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)


def mcnemar(ok_a, ok_b):
    """McNemar con corrección de continuidad sobre aciertos pareados (0/1). Devuelve b, c, chi², p."""
    b = int(((ok_a == 1) & (ok_b == 0)).sum())
    c = int(((ok_a == 0) & (ok_b == 1)).sum())
    if b + c == 0:
        return b, c, 0.0, 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return b, c, chi2, math.erfc(math.sqrt(chi2 / 2))       # p de una χ² con 1 g.l.


def run_dir_for(model, quant, prompt, suffix=""):
    return RUNS / f"zeroshot_{model}_{quant}_{prompt}_val{suffix}"

## 0. Carga de los runs

Runs de val completo `zeroshot_<modelo>_<quant>_<prompt>_val`. Se excluyen los pilotos (`_n200`) y la condición
intermedia de VL-JEPA (`_decm3di`), que se cargan en sus secciones. Puntuar 18 runs × 10 000 respuestas toma
un par de minutos.

In [ ]:
RUN_DIRS = sorted(p for p in RUNS.glob("zeroshot_*_val") if (p / "predictions.jsonl").exists())
RUN = {}
for p in RUN_DIRS:
    df = load_run(p)
    RUN[(df.attrs["model"], df.attrs["quant"], df.attrs["prompt"])] = df
    print(f"{p.name:<60} n={len(df):>6}")
incomplete = [k for k, df in RUN.items() if len(df) != 10_000]
print("\nruns con n != 10 000 (no comparables):", incomplete or "ninguno")

## 1. Líneas base sin modelo

- **Clase mayoritaria:** responder siempre el valor más frecuente de train. Es el piso que un modelo debe superar.
- **Techo del color:** el nombre de color depende del matiz *y* de una paleta elegida al azar que la imagen no
  muestra; el mejor lector posible responde, para cada matiz, el nombre más frecuente (`color_ceiling`). Ningún
  modelo puede superarlo de forma sistemática.

In [ ]:
train_recs, val_recs = load_manifest("train"), load_manifest("val")
BASE_MAJ = majority_baseline(train_recs, val_recs)
BASE_COLOR = color_ceiling(train_recs, val_recs)
base = pd.DataFrame([{**{ATTR_ES[a]: BASE_MAJ[a] for a in ATTRIBUTES}, "attr_mean": BASE_MAJ["attr_mean"],
                      "techo_color": BASE_COLOR["color_ceiling"]}], index=["clase mayoritaria"])
display(keep(base, "lineas_base", "§1"))
print(f"techo del color: {BASE_COLOR['color_ceiling']:.3f} ({BASE_COLOR['n_bins']} intervalos de matiz)")

## 2. Exactitud por atributo, modelo × prompt (NF4)

In [ ]:
S = pd.DataFrame([summary_row(df) for (m, q, p), df in RUN.items() if q == "nf4" and m in VLMS])
S["modelo"] = S["model"].map(MODELS)
tabla = keep(S.set_index(["modelo", "prompt"])[[*METRICS, *[f"omit_{a}" for a in ATTRIBUTES], "words"]],
             "modelo_x_prompt_nf4", "§2")
display(tabla.style.format("{:.3f}").background_gradient(subset=["attr_mean"], cmap="Greens"))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(19, 3.6), sharey=True)
for ax, m in zip(axes, [*ATTRIBUTES, "attr_mean"]):
    piv = S.pivot(index="modelo", columns="prompt", values=m).reindex(columns=PROMPTS)
    im = ax.imshow(piv.to_numpy(), vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(PROMPTS)), [p.split("_")[0] for p in PROMPTS], rotation=0)
    ax.set_yticks(range(len(piv)), piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.iat[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="w" if v < 0.5 else "k", fontsize=8)
    base_v = BASE_MAJ[m] if m in BASE_MAJ else None
    ax.set_title(f"{ATTR_ES[m]}" + (f"  (mayoritaria {base_v:.2f})" if base_v is not None else ""), fontsize=9)
    ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.8)
show("modelo_x_prompt_mapas", fig, "§2")

## 3. Selección de prompt y sensibilidad

Regla (`docs/metodologia.md` §2.2): por modelo, el prompt con mayor `attr_mean` en val. La selección **se reporta
pero no decide** el prompt de la etapa 2 (`p1_attributes` en los 4). Se verifica contra
`docs/resultados_etapa1.md` §4 y se estima si el margen sobre el segundo es distinguible del ruido con un
bootstrap pareado (mismas imágenes, IC 95 %).

In [ ]:
DOCUMENTED = {"qwen35_08b_base": "p1_attributes", "llava_ov_05b": "p3_json", "internvl3_1b": "p1_attributes"}
SELECTED, rows = {}, []
for m in VLMS:
    sub = S[S["model"] == m].sort_values("attr_mean", ascending=False)
    if sub.empty:
        continue
    first, second = sub.iloc[0], sub.iloc[1]
    SELECTED[m] = first["prompt"]
    a = RUN[(m, "nf4", first["prompt"])]["ok_attr_mean"]
    b = RUN[(m, "nf4", second["prompt"])]["ok_attr_mean"].reindex(a.index)
    d, lo, hi = paired_bootstrap(a, b)
    rows.append({"modelo": MODELS[m], "elegido": first["prompt"], "attr_mean": first["attr_mean"],
                 "segundo": second["prompt"], "attr_mean_2": second["attr_mean"], "margen": d,
                 "IC95_lo": lo, "IC95_hi": hi, "robusto": lo > 0,
                 "coincide_con_doc": DOCUMENTED.get(m) == first["prompt"],
                 "rango_entre_prompts": sub["attr_mean"].max() - sub["attr_mean"].min()})
display(keep(pd.DataFrame(rows).set_index("modelo"), "seleccion_prompt", "§3"))

**Lectura.** `robusto = False` significa que el IC del margen incluye el 0: la elección entre los dos primeros
prompts no se distingue del ruido de muestreo con 10 000 imágenes (caso esperado: LLaVA-OV, `p3_json` frente a
`p1_attributes`, `docs/resultados_etapa1.md` §4). `rango_entre_prompts` mide la sensibilidad al prompt del modelo.

## 4. Omitir frente a equivocarse

Para cada atributo, con el prompt elegido: fracción de respuestas **correctas**, **incorrectas** (el modelo
nombra un valor y es otro) y **omitidas** (el extractor no encuentra el atributo; cuenta como error). Una
exactitud baja por omisión es un problema de formato, no necesariamente de percepción.

In [ ]:
fig, axes = plt.subplots(1, len(SELECTED), figsize=(5 * len(SELECTED), 3.4), sharey=True)
for ax, (m, p) in zip(np.atleast_1d(axes), SELECTED.items()):
    df = RUN[(m, "nf4", p)]
    ok = np.array([df[f"ok_{a}"].mean() for a in ATTRIBUTES])
    om = np.array([df[f"pred_{a}"].isna().mean() for a in ATTRIBUTES])
    wr = 1 - ok - om
    x = np.arange(len(ATTRIBUTES))
    ax.bar(x, ok, label="correcta", color="#2a9d8f")
    ax.bar(x, wr, bottom=ok, label="incorrecta", color="#e76f51")
    ax.bar(x, om, bottom=ok + wr, label="omitida", color="#bbbbbb")
    ax.scatter(x, [BASE_MAJ[a] for a in ATTRIBUTES], marker="_", s=400, color="k", label="mayoritaria", zorder=3)
    ax.set_xticks(x, [ATTR_ES[a] for a in ATTRIBUTES])
    ax.set_title(f"{MODELS[m]} — {p}", fontsize=9)
np.atleast_1d(axes)[0].legend(fontsize=7, loc="upper right")
show("omision_vs_error", fig, "§4")

# exactitud CONDICIONAL a responder: ¿acierta cuando sí nombra el atributo?
cond = []
for m, p in SELECTED.items():
    df = RUN[(m, "nf4", p)]
    cond.append({"modelo": MODELS[m], **{ATTR_ES[a]: df.loc[df[f"pred_{a}"].notna(), f"ok_{a}"].mean() for a in ATTRIBUTES}})
print("Exactitud entre las respuestas que SÍ nombran el atributo:")
display(keep(pd.DataFrame(cond).set_index("modelo"), "exactitud_condicional", "§4"))

## 5. Matrices de confusión (prompt elegido)

Filas: valor verdadero; columnas: valor extraído (`∅` = omitido). Normalizadas por fila. Una columna dominante
indica una respuesta constante (p. ej. LLaVA-OV con `p3_json` responde `y = top` en todas las imágenes).

In [ ]:
for a in ("shape", "x", "y"):
    fig, axes = plt.subplots(1, len(SELECTED), figsize=(5.2 * len(SELECTED), 4))
    for ax, (m, p) in zip(np.atleast_1d(axes), SELECTED.items()):
        df = RUN[(m, "nf4", p)]
        ct = pd.crosstab(df[f"true_{a}"], df[f"pred_{a}"].fillna("∅"), normalize="index")
        cols = [c for c in ct.index if c in ct.columns] + [c for c in ct.columns if c not in ct.index]
        ct = ct.reindex(columns=cols, fill_value=0)
        ax.imshow(ct.to_numpy(), vmin=0, vmax=1, cmap="Blues", aspect="auto")
        ax.set_xticks(range(ct.shape[1]), ct.columns, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(ct.shape[0]), ct.index, fontsize=7)
        for i in range(ct.shape[0]):
            for j in range(ct.shape[1]):
                if ct.iat[i, j] >= 0.05:
                    ax.text(j, i, f"{ct.iat[i, j]:.2f}", ha="center", va="center", fontsize=6,
                            color="w" if ct.iat[i, j] > 0.5 else "k")
        ax.set_title(f"{ATTR_ES[a]} — {MODELS[m]} ({p})", fontsize=9)
        ax.grid(False)
    plt.tight_layout()
    show(f"confusion_{a}", fig, "§5")

## 6. Color: confusiones más frecuentes

El color se puntúa por nombre exacto (sin prefijo de paleta). Recordar el techo de §1: buena parte del error es
irreducible porque la paleta no se ve en la imagen.

In [ ]:
for m, p in SELECTED.items():
    df = RUN[(m, "nf4", p)]
    wrong = df[(df["ok_color"] == 0) & df["pred_color"].notna()]
    top = (wrong.groupby(["true_color", "pred_color"]).size().sort_values(ascending=False).head(10)
           .rename("n").reset_index())
    print(f"{MODELS[m]} ({p}): exactitud {df['ok_color'].mean():.3f} | techo {BASE_COLOR['color_ceiling']:.3f} | "
          f"omitido {df['pred_color'].isna().mean():.3f} | nombres distintos predichos {df['pred_color'].nunique()}")
    display(top)

## 7. Control bf16: NF4 frente a bf16 (prompt elegido)

Las condiciones bf16 son **controles**; toda selección y comparación principal es en NF4
(`docs/metodologia.md` §2.1). Por atributo: exactitud en cada precisión, diferencia, y McNemar sobre los aciertos
pareados (b = acierta solo NF4, c = acierta solo bf16).

In [ ]:
rows = []
for m, p in SELECTED.items():
    if (m, "bf16", p) not in RUN:
        print(f"falta el control bf16 de {MODELS[m]} ({p})")
        continue
    a, b = RUN[(m, "nf4", p)], RUN[(m, "bf16", p)]
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    for met in METRICS:
        if met == "attr_mean":
            d, lo, hi = paired_bootstrap(a["ok_attr_mean"], b["ok_attr_mean"])
            rows.append({"modelo": MODELS[m], "métrica": met, "nf4": a[f"ok_{met}"].mean(), "bf16": b[f"ok_{met}"].mean(),
                         "nf4−bf16": d, "IC95": f"[{lo:+.3f}, {hi:+.3f}]", "b": None, "c": None, "p": None})
            continue
        bb, cc, chi2, pv = mcnemar(a[f"ok_{met}"], b[f"ok_{met}"])
        rows.append({"modelo": MODELS[m], "métrica": met, "nf4": a[f"ok_{met}"].mean(), "bf16": b[f"ok_{met}"].mean(),
                     "nf4−bf16": a[f"ok_{met}"].mean() - b[f"ok_{met}"].mean(), "IC95": "", "b": bb, "c": cc, "p": pv})
    om = {f"omit_{x}": (a[f"pred_{x}"].isna().mean(), b[f"pred_{x}"].isna().mean()) for x in ("x", "y")}
    print(f"{MODELS[m]}: omisión x/y NF4 = {om['omit_x'][0]:.3f}/{om['omit_y'][0]:.3f}; "
          f"bf16 = {om['omit_x'][1]:.3f}/{om['omit_y'][1]:.3f}")
CTRL = pd.DataFrame(rows)
if not CTRL.empty:
    display(keep(CTRL.set_index(["modelo", "métrica"]), "control_bf16", "§7"))

**Lectura.** Si una diferencia NF4 − bf16 se explica por omisiones (p. ej. InternVL3 en bf16 omite la posición,
`docs/resultados_etapa1.md` §4b), es un cambio de **formato** de la respuesta, no de percepción: comparar la
exactitud condicional de §4 en ambas precisiones antes de atribuirla a la cuantización.

## 8. Métricas de texto frente a fidelidad por atributo

Las métricas de texto (BLEU, ROUGE-L, CIDEr; `pycocoevalcap`) miden parecido con el caption original; la
exactitud por atributo mide si la descripción es **correcta**. Si ambas no ordenan igual los runs, las
métricas de texto premian el formato del dataset y no la fidelidad (argumento central del proyecto).

In [ ]:
T = []
for (m, q, p), df in RUN.items():
    if q != "nf4" or m not in VLMS:
        continue
    met = run_metrics(run_dir_for(m, q, p))
    if met is None or not all(k in met for k in TEXT_METRICS):
        print(f"sin métricas de texto: {m} {p} (correr scripts/score.py)")
        continue
    T.append({"modelo": MODELS[m], "prompt": p, "attr_mean": df["ok_attr_mean"].mean(), **{k: met[k] for k in TEXT_METRICS}})
T = pd.DataFrame(T)
if not T.empty:
    keep(T, "texto_vs_atributos", "§8")
    display(keep(T.corr(method="spearman", numeric_only=True).loc[["attr_mean"], list(TEXT_METRICS)]
                 .rename(index={"attr_mean": "Spearman con attr_mean"}), "texto_vs_atributos_spearman", "§8"))
    fig, axes = plt.subplots(1, len(TEXT_METRICS), figsize=(4.2 * len(TEXT_METRICS), 3.4), sharey=True)
    for ax, k in zip(axes, TEXT_METRICS):
        for mod, g in T.groupby("modelo"):
            ax.scatter(g[k], g["attr_mean"], label=mod, s=30)
            for _, r in g.iterrows():
                ax.annotate(r["prompt"].split("_")[0], (r[k], r["attr_mean"]), fontsize=6, alpha=0.7)
        ax.set_xlabel(k)
    axes[0].set_ylabel("attr_mean")
    axes[0].legend(fontsize=7)
    show("texto_vs_atributos", fig, "§8")

## 9. Piloto (200 imágenes) frente a val completo

¿El piloto de 200 imágenes anticipaba el ordenamiento de prompts del val completo? Se compara la `attr_mean`
de cada (modelo, prompt) y el prompt que cada uno habría elegido.

In [ ]:
P = []
for p_dir in sorted(RUNS.glob("zeroshot_*_nf4_*_val_n200")):
    if not (p_dir / "predictions.jsonl").exists():
        continue
    d200 = load_run(p_dir)
    key = (d200.attrs["model"], "nf4", d200.attrs["prompt"])
    if key in RUN and key[0] in VLMS:
        P.append({"modelo": MODELS[key[0]], "prompt": key[2], "piloto_200": d200["ok_attr_mean"].mean(),
                  "val_10000": RUN[key]["ok_attr_mean"].mean()})
P = pd.DataFrame(P)
if not P.empty:
    agree = []
    for mod, g in P.groupby("modelo"):
        agree.append({"modelo": mod, "Spearman": g[["piloto_200", "val_10000"]].corr(method="spearman").iat[0, 1],
                      "elegido_piloto": g.loc[g["piloto_200"].idxmax(), "prompt"],
                      "elegido_val": g.loc[g["val_10000"].idxmax(), "prompt"]})
    display(keep(pd.DataFrame(agree).set_index("modelo"), "piloto_vs_val", "§9"))
    ax = P.plot.scatter("piloto_200", "val_10000", figsize=(4.5, 4))
    lim = [0, max(P[["piloto_200", "val_10000"]].max()) + 0.05]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_title("attr_mean: piloto vs val completo", fontsize=9)
    show("piloto_vs_val", ax.figure, "§9")

## 10. Y-Decoder genérico (MSR-VTT)

El Y-Decoder (diseño propio, `docs/metodologia.md` §2.4) convierte el embedding de VL-JEPA en texto. El genérico
se entrena **solo con captions de MSR-VTT** y el paso se elige por CIDEr en MSR-VTT `test_1k`; el techo en M3DI
(`m3di_*`: decodificar el `S_Y` del caption verdadero) **solo se reporta**. Barrido de *k* (vectores de prefijo) con
1 época; el ganador (k = 8) se entrena hasta 3 épocas.

In [ ]:
YD = {}
for d in sorted(RUNS.glob("ydec_vljepa*_msrvtt_k*_r64_e*")):
    log = d / "val_log.jsonl"
    if log.exists() and "humo" not in d.name:
        YD[d.name] = pd.DataFrame(read_jsonl(log))
if YD:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for name, df in YD.items():
        lab = name.split("_msrvtt_")[-1] + (" (bf16)" if "_bf16_" in name else "")
        axes[0].plot(df["step"], df["cider"], marker=".", label=lab)
        if "m3di_attr_mean" in df:
            axes[1].plot(df["step"], df["m3di_attr_mean"], marker=".", label=lab)
    axes[0].set(title="CIDEr en MSR-VTT test_1k (elige el paso)", xlabel="paso")
    axes[1].set(title="techo en M3DI, attr_mean (solo reporte)", xlabel="paso")
    axes[0].legend(fontsize=7)
    show("ydecoder_generico", fig, "§10")
    best = []
    for name, df in YD.items():
        b = df.loc[df["cider"].idxmax()]
        best.append({"run": name, "mejor_paso": int(b["step"]), "cider": b["cider"], "bleu4": b.get("bleu4"),
                     **{f"m3di_{a}": b.get(f"m3di_{a}") for a in [*ATTRIBUTES, "attr_mean"]}})
    display(keep(pd.DataFrame(best).set_index("run"), "ydecoder_generico", "§10"))
else:
    print("no hay runs del Y-Decoder genérico")

## 11. VL-JEPA (si ya hay runs)

### 11a. Zero-shot con el Y-Decoder genérico, por prompt

Con el lector genérico, el techo de VL-JEPA en M3DI es ≈ 0.1: el decodificador no escribe posiciones ni
nombres de paleta. Una exactitud baja aquí es de **vocabulario del lector**, no necesariamente de percepción;
la condición intermedia (11b) y las sondas (11d), que lo separarían, se cancelaron en el cierre (CE1); lo más
informativo que queda es el diagnóstico de embeddings (11c).

In [ ]:
VJ = "vljepa_open_msrvtt"
vj = {p: RUN[(VJ, "nf4", p)] for p in PROMPTS if (VJ, "nf4", p) in RUN}
if vj:
    VJS = pd.DataFrame([summary_row(df) for df in vj.values()]).set_index("prompt")
    display(keep(VJS[[*METRICS, *[f"omit_{a}" for a in ATTRIBUTES], "words"]], "vljepa_zeroshot_prompts", "§11a"))
    SELECTED[VJ] = VJS["attr_mean"].idxmax()
    print("prompt elegido para VL-JEPA (attr_mean):", SELECTED[VJ])
    for _, r in vj[SELECTED[VJ]].sample(5, random_state=0).iterrows():
        print(f"  ref: {r['reference']}\n  dec: {r['prediction']}\n")
else:
    print("VL-JEPA zero-shot en val todavía no existe (configs/queues/stage1_vljepa_val.txt)")

### 11b. Condición intermedia y control bf16 de VL-JEPA

**Cancelados en el cierre de la etapa 1 (CE1).** Sus decodificadores (intermedio; genérico bf16) no se entrenaron.
La celda queda para que el notebook siga funcionando si algún día se corren.


In [ ]:
extra = {}
for d in sorted(RUNS.glob(f"zeroshot_{VJ}_*_val_decm3di")) + sorted(RUNS.glob(f"zeroshot_{VJ}_bf16_*_val")):
    if (d / "predictions.jsonl").exists():
        extra[d.name] = summary_row(load_run(d))
if extra:
    display(pd.DataFrame(extra).T[["quant", "prompt", "n", *METRICS]])
else:
    print("sin condición intermedia ni control bf16 de VL-JEPA todavía")

### 11c. Diagnóstico de cuantización (`scripts/diag_quant_vljepa.py`)

Alineación (coseno Ŝ_Y–S_Y), discriminación (recall@k, InfoNCE en lotes fijos de 128; azar = log 128 ≈ 4.85) y
coseno de cada etapa contra bf16. Si un componente explica casi toda la caída de NF4, se revisa su cuantización.

In [ ]:
DQ = []
for d in sorted(RUNS.glob("diagquant_*")):
    if "humo" in d.name or not (d / "metrics.json").exists():
        continue
    m = json.loads((d / "metrics.json").read_text(encoding="utf-8"))
    disc, al = m["discrimination"], m["alignment"]["cos_pred_target"]
    row = {"condición": m["condition"], "cos_mean": al["mean"], "cos_p10": al["p10"],
           "R@1 p→t": disc["recall@1_p2t"], "R@1 t→p": disc["recall@1_t2p"], "R@5 p→t": disc["recall@5_p2t"],
           "azar@1": disc["chance@1"], "InfoNCE": disc["infonce"], "acc_lote": disc["infonce_batch_acc"]}
    if isinstance(m.get("vs_bf16"), dict):
        row |= {f"vs_bf16 {k.split('_')[0]}": v["mean"] for k, v in m["vs_bf16"].items() if isinstance(v, dict)}
        row["Δcos"] = m["vs_bf16"].get("delta_cos_pred_target")
    DQ.append(row)
if DQ:
    order = ["bf16", "nf4", "nf4_x_encoder", "nf4_predictor", "nf4_y_encoder"]
    DQ = pd.DataFrame(DQ).set_index("condición")
    display(keep(DQ.reindex([o for o in order if o in DQ.index] + [i for i in DQ.index if i not in order]),
                 "vljepa_diag_cuantizacion", "§11c"))
else:
    print("el diagnóstico de cuantización todavía no se ha corrido (configs/queues/diag_quant_vljepa.txt)")

#### 11c-bis. ¿El coseno Ŝ_Y–S_Y distingue imágenes? (propio frente a otros)

En el diagnóstico, R@1 ≈ azar y la InfoNCE queda sobre ln 128 incluso en bf16, con un coseno medio de 0.68 con el
caption propio. Hipótesis: los captions de M3DI (plantillas casi iguales) dejan todos los S_Y muy juntos y Ŝ_Y cae
cerca del centro de ese grupo. Se compara, en las mismas 1 024 imágenes:

- **propio**: cos(Ŝ_Y_i, S_Y_i); **otros**: cos(Ŝ_Y_i, S_Y_j), i ≠ j. Si son iguales, el coseno no tiene
  información de *qué* imagen es.
- **S_Y–S_Y**: similitud media entre captions distintos; **pred–pred**: entre predicciones (≈ 1 = colapso).
- **rango mediano** del caption propio entre los 1 024 (azar ≈ 512).


In [ ]:
PO = []
for c in ["bf16", "nf4", "nf4_x_encoder", "nf4_predictor", "nf4_y_encoder"]:
    f = RUNS / f"diagquant_vljepa_open_msrvtt_{c}" / "features.npz"
    if not f.exists():
        continue
    z = np.load(f)
    p, s = z["pred"].astype(np.float64), z["s_y"].astype(np.float64)
    p /= np.linalg.norm(p, axis=1, keepdims=True)
    s /= np.linalg.norm(s, axis=1, keepdims=True)
    S = p @ s.T
    off = ~np.eye(len(S), dtype=bool)
    rank = (S > np.diag(S)[:, None]).sum(1)
    PO.append({"condición": c, "propio": np.diag(S).mean(), "otros": S[off].mean(), "otros_sd": S[off].std(),
               "propio − otros": np.diag(S).mean() - S[off].mean(), "S_Y–S_Y": (s @ s.T)[off].mean(),
               "pred–pred": (p @ p.T)[off].mean(), "rango_mediano": int(np.median(rank)), "n": len(S)})
if PO:
    display(keep(pd.DataFrame(PO).set_index("condición"), "vljepa_propio_vs_otros", "§11c-bis"))
else:
    print("sin features.npz del diagnóstico (configs/queues/diag_quant_vljepa.txt)")


### 11d. Pérdida de información por etapa (`scripts/info_loss.py`)

**Cancelada en la etapa 1 (CE1).** El comando sigue en `configs/queues/stage1_infoloss.txt` (solo inferencia) por
si la etapa 2 necesita la comparación. Exactitud de sondas por factor en E1 (X-Encoder), E2 (Predictor), E3 (Ŝ_Y),
T1 (S_Y) y, con decodificador, E4/T2 (texto) (`docs/metodologia.md` §2.5).


In [ ]:
found = False
for d in sorted(RUNS.glob("infoloss_*")):
    t = d / "tabla_discretos.csv"
    if t.exists() and "humo" not in d.name:
        found = True
        print(d.name)
        display(pd.read_csv(t))
if not found:
    print("la pérdida de información de la etapa 1 todavía no se ha corrido (configs/queues/stage1_infoloss.txt)")

## 12. Tabla resumen para el informe (val, prompt elegido por modelo)

NF4 es la condición principal; bf16, el control. La comparación válida es **entre nuestros modelos bajo el mismo
protocolo**, no contra cifras de los papers (que usan modelos de 7B o más).

In [ ]:
R = []
for m, p in SELECTED.items():
    for q in ("nf4", "bf16"):
        if (m, q, p) not in RUN:
            continue
        df = RUN[(m, q, p)]
        met = run_metrics(run_dir_for(m, q, p)) or {}
        R.append({"modelo": MODELS[m], "precisión": q, "prompt": p, **{ATTR_ES[x]: df[f"ok_{x}"].mean() for x in METRICS},
                  **{k: met.get(k) for k in TEXT_METRICS}})
R.append({"modelo": "clase mayoritaria", "precisión": "—", "prompt": "—",
          **{ATTR_ES[a]: BASE_MAJ[a] for a in ATTRIBUTES}, "attr_mean": BASE_MAJ["attr_mean"]})
RES = pd.DataFrame(R).set_index(["modelo", "precisión"])
display(keep(RES, "etapa1_resumen", "§12"))